<a href="https://colab.research.google.com/github/ibrahimhanifceker/YZ50Codes/blob/main/hafta-5/YZ50Hafta5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Forward Pass Parcalama

In [211]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [212]:
import os

if not os.path.exists("names.txt"):
    !wget https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt

words = open("names.txt", 'r').read().splitlines()

In [213]:
chars = ['.'] + list(sorted(set(''.join(words))))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)

In [214]:
block_size = 3

def build_dataset(words):
    X, Y = [], []

    for word in words:
        context = [0] * block_size
        for ch in word + '.':
            idx = stoi[ch]
            X.append(context)
            Y.append(idx)
            context = context[1:] + [idx]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
split_1 = int(0.8 * len(words))
split_2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:split_1])
Xdev, Ydev = build_dataset(words[split_1:split_2])
Xte, Yte = build_dataset(words[split_2:])

In [215]:
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f"{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}")

In [216]:
n_embd = 10
n_hidden = 64

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * 5/3 * (n_embd * block_size)**-0.5
b1 = torch.randn(n_hidden, generator=g) * 0.1
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1

bngain = torch.randn((1, n_hidden), generator=g) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden), generator=g) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for param in parameters:
    param.requires_grad = True

In [217]:
batch_size = 32
n = batch_size
idx = torch.randint(0, len(Xtr), (n,), generator=g)
Xb, Yb = Xtr[idx], Ytr[idx]

In [218]:
emb = C[Xb]
embcat = emb.reshape(n, -1)
hprebn = embcat @ W1 + b1
bnmeani = 1/n * hprebn.sum(dim=0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n - 1) * bndiff2.sum(dim=0, keepdim=True)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
h = torch.tanh(hpreact)
logits = h @ W2 + b2

logit_maxes = logits.max(dim=1, keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(dim=1, keepdim=True)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

for p in parameters:
    p.grad = None

for t in [logprobs, probs, counts_sum_inv, probs, counts_sum_inv, counts_sum, counts, norm_logits, logit_maxes, logits, h, hpreact, bnraw, bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani, embcat, emb]:
    t.retain_grad()
loss.backward()
loss

tensor(3.5571, grad_fn=<NegBackward0>)

# 2. Manuel Gradient Yazma

In [219]:
dlogprobs = -1/n * F.one_hot(Yb, num_classes=vocab_size).float()
dprobs = dlogprobs * probs**-1
dcounts_sum_inv = (dprobs * counts).sum(dim=1, keepdim=True)
dcounts_sum = dcounts_sum_inv * -counts_sum**-2
dcounts = dprobs * counts_sum_inv + dcounts_sum
dnorm_logits = dcounts * counts
dlogit_maxes = -dnorm_logits.sum(dim=1, keepdim=True)
dlogits = dnorm_logits + dlogit_maxes * F.one_hot(logits.max(dim=1).indices, num_classes=vocab_size)
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(dim=0)
dhpreact = dh * (1 - h**2)
dbngain = (dhpreact * bnraw).sum(dim=0, keepdim=True)
dbnbias = dhpreact.sum(dim=0, keepdim=True)
dbnraw = dhpreact * bngain
dbnvar_inv = (dbnraw * bndiff).sum(dim=0, keepdim=True)
dbnvar = dbnvar_inv * -0.5 * (bnvar + 1e-5)**-1.5
dbndiff2 = 1/(n - 1) * dbnvar.expand_as(bndiff2)
dbndiff = dbnraw * bnvar_inv + 2 * bndiff * dbndiff2
dbnmeani = -dbndiff.sum(dim=0, keepdim=True)
dhprebn = dbndiff + 1/n * dbnmeani
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(dim=0)
demb = dembcat.reshape(n, block_size, -1)
dC = torch.zeros((vocab_size, n_embd))
dC.index_add_(dim=0, index=Xb.reshape(-1), source=demb.reshape(-1, n_embd))

cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: False | approximate: True  | maxdiff: 4.656612873077393e-10
bngain          | exact: False | approximate: True  | maxdiff: 1.3969838619232178e-09
bnbias          | exact: False | approximate: True  | maxdiff: 3.725290298461914e-09
bnraw  

# 3. Cross Entropy ve Batch Norm Optimize

In [220]:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.5571467876434326 diff: -2.384185791015625e-07


In [221]:
probs = F.softmax(logits, dim=1)
dlogits = 1/n * (probs - F.one_hot(Yb, num_classes=vocab_size))

cmp('logits', dlogits, logits)

logits          | exact: False | approximate: True  | maxdiff: 5.587935447692871e-09


In [222]:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

max diff: tensor(4.7684e-07, grad_fn=<MaxBackward1>)


In [223]:
dhprebn = bngain * (dhpreact * bnvar_inv - 1/n * bnvar_inv * dhpreact.sum(0, keepdim=True) - 1/(n - 1) * bndiff * bnvar_inv**3 * (dhpreact * bndiff).sum(0, keepdim=True))
cmp('hprebn', dhprebn, hprebn)

hprebn          | exact: False | approximate: True  | maxdiff: 1.3969838619232178e-09


Training

In [241]:
n_embd = 10
n_hidden = 200

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * 5/3 * (n_embd * block_size)**-0.5
b1 = torch.randn(n_hidden, generator=g) * 0.1
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1

bngain = torch.randn((1, n_hidden), generator=g) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden), generator=g) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for param in parameters:
    param.requires_grad = True

epochs = 200000
batch_size = 32
n = batch_size
loss_vals = []

with torch.inference_mode():
    for epoch in range(epochs):
        idx = torch.randint(0, len(Xtr), (n,), generator=g)
        Xb, Yb = Xtr[idx], Ytr[idx]
        emb = C[Xb]
        embcat = emb.reshape(n, -1)
        hprebn = embcat @ W1 + b1
        bnmean = 1/n * hprebn.sum(dim=0, keepdim=True)
        bnvar = hprebn.var(dim=0, keepdim=True, unbiased=True)
        bnvar_inv = (bnvar + 1e-5)**-0.5
        bnraw = (hprebn - bnmean) * bnvar_inv
        hpreact = bngain * bnraw + bnbias
        h = torch.tanh(hpreact)
        logits = h @ W2 + b2

        loss = F.cross_entropy(logits, Yb)

        for p in parameters:
            p.grad = None

        # loss.backward()

        dlogits = 1/n * (F.softmax(logits, dim=1) - F.one_hot(Yb, num_classes=vocab_size))
        dh = dlogits @ W2.T
        dW2 = h.T @ dlogits
        db2 = dlogits.sum(dim=0)
        dhpreact = dh * (1 - h**2)
        dbngain = (dhpreact * bnraw).sum(dim=0, keepdim=True)
        dbnbias = dhpreact.sum(dim=0, keepdim=True)
        dhprebn = bngain * (dhpreact * bnvar_inv - 1/n * bnvar_inv * dhpreact.sum(0, keepdim=True) - 1/(n - 1) * (hprebn - bnmean) * bnvar_inv**3 * (dhpreact * (hprebn - bnmean)).sum(0, keepdim=True))
        dembcat = dhprebn @ W1.T
        dW1 = embcat.T @ dhprebn
        db1 = dhprebn.sum(dim=0)
        demb = dembcat.reshape(n, block_size, -1)
        dC = torch.zeros((vocab_size, n_embd))
        dC.index_add_(dim=0, index=Xb.reshape(-1), source=demb.reshape(-1, n_embd))

        lr = 0.1 if epoch < epochs / 2 else 0.01

        grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]

        for param, grad in zip(parameters, grads):
            # param.data += -lr * param.grad
            param.data += -lr * grad

        if epoch % 10000 == 0:
            print(f"Epoch {epoch}/{epochs} | Loss: {loss.item()}")

Epoch 0/200000 | Loss: 3.950787305831909
Epoch 10000/200000 | Loss: 2.3617022037506104
Epoch 20000/200000 | Loss: 2.297894239425659
Epoch 30000/200000 | Loss: 2.219085454940796
Epoch 40000/200000 | Loss: 2.543769359588623
Epoch 50000/200000 | Loss: 2.136763572692871
Epoch 60000/200000 | Loss: 2.2738091945648193
Epoch 70000/200000 | Loss: 1.9721814393997192
Epoch 80000/200000 | Loss: 1.8331637382507324
Epoch 90000/200000 | Loss: 2.5328660011291504
Epoch 100000/200000 | Loss: 2.199350595474243
Epoch 110000/200000 | Loss: 2.3767054080963135
Epoch 120000/200000 | Loss: 2.365363597869873
Epoch 130000/200000 | Loss: 2.4308278560638428
Epoch 140000/200000 | Loss: 2.339050769805908
Epoch 150000/200000 | Loss: 2.119284152984619
Epoch 160000/200000 | Loss: 2.0435736179351807
Epoch 170000/200000 | Loss: 2.0773441791534424
Epoch 180000/200000 | Loss: 2.0449187755584717
Epoch 190000/200000 | Loss: 2.2389562129974365


In [254]:
with torch.inference_mode():
    emb = C[Xtr]
    embcat = emb.reshape(len(Xtr), -1)
    hprebn = embcat @ W1 + b1
    bnmean_all = hprebn.mean(dim=0, keepdim=True)
    bnvar_all = hprebn.var(dim=0, keepdim=True, unbiased=True)

In [255]:
@torch.inference_mode()
def get_loss(split):
    x, y = {
        "train": {Xtr, Ytr},
        "dev": {Xdev, Ydev},
        "test": {Xte, Yte},
    }[split]
    emb = C[x]
    embcat = emb.reshape(len(x), -1)
    hprebn = embcat @ W1
    hpreact = bngain * (hprebn - bnmean_all) * (bnvar_all + 1e-5)**-0.5 + bnbias
    hidden_layer = torch.tanh(hpreact)
    logits = hidden_layer @ W2 + b2
    loss = F.cross_entropy(logits, y)
    return loss.item()

get_loss("train"), get_loss("dev"), get_loss("test")

(2.0786244869232178, 2.1193971633911133, 2.1183245182037354)

In [257]:
num_names = 20

for _ in range(num_names):
    context = [0] * block_size
    name = []
    while True:
        emb = C[torch.tensor(context)]
        embcat = emb.reshape(1, -1)
        hprebn = embcat @ W1 + b1
        hpreact = bngain * (hprebn - bnmean_all) * (bnvar_all + 1e-5)**-0.5 + bnbias
        h = torch.tanh(hpreact)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        idx = torch.multinomial(probs, num_samples=1, generator=g).item()
        if idx == 0:
            break
        context = context[1:] + [idx]
        name.append(itos[idx])
    print(''.join(name))

zuraphnarah
zika
saello
fiten
cloylan
vion
lania
ell
jama
haddiah
avrae
harleri
adrickellen
jani
roslayla
voek
yosala
caton
traxtonni
arrusadalakeinnon
